# 基本介绍

```python
def search(
    self,
    namespace_prefix: tuple[str, ...],
    /,
    *,
    query: str | None = None,
    filter: dict[str, Any] | None = None,
    limit: int = 10,
    offset: int = 0,
    refresh_ttl: bool | None = None,
) -> list[SearchItem]:
```

### 参数说明
- `namespace_prefix`: 命名空间前缀，在该前缀下搜索。
- `query`: 语义检索时用于查询的自然语言。
- `filter`: 过滤条件，value中的键值对组合，见下文举例。
- `limit`: 可以返回item的最大条数，效果等同于SQL中的limit。
- `offset`: 返回结果之前跳过的item数量。
- `refresh_ttl`: 同上。

它支持两种检索方式（对应上面的参数2、3）：
- 按 `filter` 做结构化过滤，即用 value 中的键值筛选符合条件的记录。也就是关键词搜索。
- 按 `query` 做语义相似度检索，需要将输入转换为向量。向量搜索。

### 返回值
返回匹配的 `SearchItem` 列表，并额外携带匹配分数等检索元信息。

In [1]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

# 示例1-按照namespace前缀搜索

In [2]:
print('=' * 30, '-> (users, ) <-', "=" * 30)
for item in store.search(("users", )):
    print(item)

============================== -> (users, ) <- ==============================
Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-09-01T07:43:48.746219+00:00', updated_at='2026-09-01T07:43:48.746221+00:00', score=None)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-09-01T07:43:48.746248+00:00', updated_at='2026-09-01T07:43:48.746248+00:00', score=None)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-09-01T07:43:48.746272+00:00', updated_at='2026-09-01T07:43:48.746272+00:00', score=None)


# 示例2-按照filter过滤

In [3]:
print("=" * 30, '-> (users, ), filter=sports <-", "=" * 30)')
for item in store.search(("users", ), filter={"sports": "跑步"}):
    print(item)

============================== -> (users, ), filter=sports <-", "=" * 30)
Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-09-01T07:43:48.746219+00:00', updated_at='2026-09-01T07:43:48.746221+00:00', score=None)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-09-01T07:43:48.746248+00:00', updated_at='2026-09-01T07:43:48.746248+00:00', score=None)


In [4]:
print("=" * 30, '-> (users, ), filter=food <-", "=" * 30)')
for item in store.search(("users", ), filter={"food": "紫光园奶皮子酸奶"}):
    print(item)

============================== -> (users, ), filter=food <-", "=" * 30)
Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-09-01T07:43:48.746219+00:00', updated_at='2026-09-01T07:43:48.746221+00:00', score=None)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-09-01T07:43:48.746272+00:00', updated_at='2026-09-01T07:43:48.746272+00:00', score=None)
